# Classificação de Imagens com ResNet50 — Histórico e Galeria

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import numpy as np
import pandas as pd
import os
from IPython.display import display, clear_output

model = ResNet50(weights='imagenet')
historico_resultados = []
os.makedirs("uploads", exist_ok=True)

def load_and_preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    return preprocess_input(img_array)

def extract_label(filename):
    return "gato" if filename.lower().startswith("gato") else "nao_gato"

print("Setup pronto!")

In [ ]:
try:
    from google.colab import files
    print("Selecione a imagem:")
    uploaded = files.upload()
    image_files = list(uploaded.keys())
except ImportError:
    image_files = [f for f in os.listdir("uploads") if f.lower().endswith(('.webp', '.jpg', '.jpeg', '.png'))]
    print(f"Usando imagens locais de uploads/: {image_files}")

cat_labels = [
    'tabby', 'tiger_cat', 'persian_cat', 'siamese_cat',
    'egyptian_cat', 'cat', 'domestic_cat'
]

for filename in image_files:
    img_path = os.path.join("uploads", filename)

    if not os.path.exists(img_path):
        os.replace(filename, img_path)

    try:
        img_array = load_and_preprocess_image(img_path)
        preds = model.predict(img_array, verbose=0)
        decoded = decode_predictions(preds, top=1)[0]

        label_imagenet = decoded[0][1]
        score = decoded[0][2]

        pred_gato = "gato" if any(cat in label_imagenet.lower() for cat in cat_labels) else "nao_gato"
        true_label = extract_label(filename)

        if true_label == "gato":
            res = "TP" if pred_gato == "gato" else "FN"
        else:
            res = "TN" if pred_gato == "nao_gato" else "FP"

        historico_resultados.append({
            "Imagem": filename,
            "Rotulo Real": true_label,
            "Predicao": pred_gato,
            "Resultado": res
        })

        print(f"{filename} processado com sucesso!")

    except Exception as e:
        print(f"Erro em {filename}: {e}")

In [ ]:
if not historico_resultados:
    print("A tabela esta vazia. Suba imagens na Celula 2.")
else:
    clear_output(wait=True)
    df = pd.DataFrame(historico_resultados)
    display(df)
    print(f"\nTotal de analises no historico: {len(historico_resultados)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import math
import os

def visualizar_galeria(historico, imagens_por_linha=8):
    if not historico:
        print("O historico esta vazio. Suba imagens primeiro!")
        return

    itens_validos = []
    for item in historico:
        caminho = os.path.join("uploads", item['Imagem'])
        if os.path.exists(caminho):
            itens_validos.append(item)
        else:
            print(f"Arquivo nao encontrado: {item['Imagem']}")

    if not itens_validos:
        print("Nenhuma imagem encontrada na pasta 'uploads'.")
        return

    total = len(itens_validos)
    colunas = min(total, imagens_por_linha)
    linhas = math.ceil(total / colunas)

    fig, axes = plt.subplots(linhas, colunas, figsize=(2.5 * colunas, 2.5 * linhas))

    if total == 1:
        axes = np.array([axes])

    axes = axes.flatten()

    for i in range(len(axes)):
        if i < total:
            item = itens_validos[i]
            img_path = os.path.join("uploads", item['Imagem'])
            img = Image.open(img_path)

            axes[i].imshow(img)

            cor = 'red' if item['Resultado'] in ['FP', 'FN'] else 'green'

            titulo = f"{item['Imagem']}\nR:{item['Rotulo Real']} | P:{item['Predicao']}\n[{item['Resultado']}]"
            axes[i].set_title(titulo, color=cor, fontsize=9, fontweight='bold')
            axes[i].axis('off')
        else:
            axes[i].axis('off')

    plt.tight_layout()
    plt.show()

visualizar_galeria(historico_resultados)

In [ ]:
import shutil
import os
from IPython.display import clear_output

historico_resultados = []

if os.path.exists("uploads"):
    shutil.rmtree("uploads")
    os.makedirs("uploads")

clear_output()

print("SISTEMA RESETADO!")
print("A tabela agora esta vazia e a pasta de imagens foi limpa. Pode comecar novamente pela Celula 2.")